## Breakpoint for attempt number 2

In [ ]:
import numpy as np
import pandas as pd

# CSV files exported from Earth Engine (same folder as this notebook)
FILES = {
    'era5land': '/Users/sophiethomas/Desktop/MDP/Climatology/Sisal_ERA5Land_daily_all.csv',
    'era5':     '/Users/sophiethomas/Desktop/MDP/Climatology/Sisal_ERA5_daily_all.csv',
    'hycom':    '/Users/sophiethomas/Desktop/MDP/Climatology/Sisal_HYCOM_daily_all.csv',
}

# kind:    'linear'   ordinary variable (averaged)
#          'flux'     daily amount, e.g. mm/day (summed to monthly totals)
#          'circular' direction in degrees (averaged as a vector)
# monthly: optional override of the monthly aggregation ('mean', 'sum', 'max')
# Units are already converted in the Earth Engine export.
VARIABLES = {
    # ERA5-Land, two inland cells (choose CELL = 'c1' or 'c2')
    't2m_c':       dict(src='era5land', label='2 m air temperature (daily mean)', unit='°C',   kind='linear'),
    't2m_min_c':   dict(src='era5land', label='2 m air temperature (daily min)',  unit='°C',   kind='linear'),
    't2m_max_c':   dict(src='era5land', label='2 m air temperature (daily max)',  unit='°C',   kind='linear'),
    'skin_t_c':    dict(src='era5land', label='Skin temperature',                  unit='°C',   kind='linear'),
    'precip_mm':   dict(src='era5land', label='Precipitation',                     unit='mm',   kind='flux'),
    'evap_mm':     dict(src='era5land', label='Actual evaporation',                unit='mm',   kind='flux'),
    'pot_evap_mm': dict(src='era5land', label='Potential evaporation',             unit='mm',   kind='flux'),
    'evap_gap_mm': dict(src='era5land', label='Evaporation gap (potential - actual)', unit='mm', kind='flux'),
    # ERA5, sea cell
    'wind_u_ms':       dict(src='era5', label='10 m wind, eastward component',  unit='m/s',  kind='linear'),
    'wind_v_ms':       dict(src='era5', label='10 m wind, northward component', unit='m/s',  kind='linear'),
    'wind_speed_ms':   dict(src='era5', label='10 m wind speed',                unit='m/s',  kind='linear'),
    'sst_c':           dict(src='era5', label='Sea surface temperature (ERA5)', unit='°C',   kind='linear'),
    'hs_mean_m':       dict(src='era5', label='Significant wave height (mean)', unit='m',    kind='linear'),
    'hs_max_m':        dict(src='era5', label='Significant wave height (max)',  unit='m',    kind='linear', monthly='max'),
    'tp_mean_s':       dict(src='era5', label='Peak wave period',               unit='s',    kind='linear'),
    'wave_dir_deg':    dict(src='era5', label='Mean wave direction (from)',     unit='°',    kind='circular'),
    'wave_power_kw_m': dict(src='era5', label='Wave power',                     unit='kW/m', kind='linear'),
    # HYCOM, one pixel
    'temp_surface_c':      dict(src='hycom', label='Sea surface temperature (HYCOM)', unit='°C',  kind='linear'),
    'temp_bottom_c':       dict(src='hycom', label='Bottom temperature',              unit='°C',  kind='linear'),
    'salinity_surface_psu':dict(src='hycom', label='Surface salinity',                unit='PSU', kind='linear'),
    'salinity_bottom_psu': dict(src='hycom', label='Bottom salinity',                 unit='PSU', kind='linear'),
}


def load_series(var, cell='c1'):
    """Daily series on a complete daily index (missing days are NaN)."""
    info = VARIABLES[var]
    path = FILES[info['src']]
    df = pd.read_csv(path, parse_dates=['date'])
    col = f'{var}_{cell}' if info['src'] == 'era5land' else var
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found in {path}. Available: {list(df.columns)}")

    if info['src'] == 'hycom' and 'bottom_level_m' in df.columns:
        levels = df['bottom_level_m'].dropna().unique()
        print(f'HYCOM bottom level(s) used: {sorted(levels)} m'
              + ('  <-- should be a single value' if len(levels) > 1 else ''))

    s = df.set_index('date')[col].sort_index()
    s = s[~s.index.duplicated()]
    s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq='D'))
    if var.startswith('salinity'):
        s[s < 30] = np.nan      # HYCOM fill values (raw 0 -> 20 PSU)
    s.name = var
    # if VAR.startswith('salinity'):
    #    s = s['1993':'2012']        # one HYCOM model version only

    n_years = (s.index[-1] - s.index[0]).days / 365.25
    print(f'{info["label"]} [{col}]: {s.index[0].date()} to {s.index[-1].date()}, '
          f'{n_years:.1f} years, {s.isna().sum()} missing days')
    if n_years < 20:
        print(f'WARNING: only {n_years:.0f} years of data. Trends and percentiles are '
              'not robust; use this run to test the pipeline, not to report results.')
    return s, info


In [ ]:
from scipy import stats

VAR  = 't2m_min_c'      # any key from VARIABLES except 'wave_dir_deg'
CELL = 'c1'         # ERA5-Land only: rerun with 'c2' to cross-check
WINDOW, SMOOTH = 11, 31    # days: percentile window and smoothing
DETREND = False             # None = automatic (only if >= 20 years of data)

s, info = load_series(VAR, CELL)
if info['kind'] == 'circular':
    raise ValueError('Directions cannot be detrended or put into percentiles. '
                     'Use Climatology_plots.ipynb for wave direction.')

TAG   = CELL if info['src'] == 'era5land' else info['src']
UNIT  = info['unit'] + ('/day' if info['kind'] == 'flux' else '')
valid = s.dropna()
YEAR0, YEAR1 = s.index[0].year, s.index[-1].year
N_YEARS = (s.index[-1] - s.index[0]).days / 365.25
if DETREND is None:
    DETREND = N_YEARS >= 20

# --- 1. linear trend on daily anomalies (seasonal cycle removed) -----------
# --- day-of-year index, leap years collapsed onto a 365-day calendar -------
def doy365(idx):
    doy = idx.dayofyear.values.copy()
    doy[idx.is_leap_year & (doy > 59)] -= 1      # 29 Feb shares day 59
    return doy

doy_all  = doy365(s.index)
raw_clim = pd.Series(s.values, index=doy_all).groupby(level=0).mean()
anom     = s - raw_clim.reindex(doy_all).values

t_all = (s.index - s.index[0]).days.values / 365.25
t     = t_all[anom.notna().values]
slope, icpt = np.polyfit(t, anom.dropna().values, 1)
print(f'daily-fit trend: {slope*10:.3f} {UNIT} per decade')

# --- 2. trend with uncertainty, on annual means (years >= 90% complete) ----
g   = s.resample('YE')
ann = g.mean()[g.count() >= 329]
if len(ann) >= 3:
    res = stats.linregress(ann.index.year, ann.values)
    print(f'annual-fit trend: {res.slope*10:.3f} '
          f'± {res.stderr*10*1.96:.3f} {UNIT} per decade (95%)')
    print('Mann-Kendall:', stats.kendalltau(ann.index.year, ann.values))
else:
    res = None
    print('Too few complete years for an annual trend.')

# --- 3. detrend (keeps the mean level) --------------------------------------
x = s - slope * (t_all - t.mean()) if DETREND else s.copy()
print('detrended before climatology:', DETREND)

# --- 4. day-of-year index, leap years collapsed onto a 365-day calendar ----
doy = doy365(x.index)
r   = pd.Series(x.values, index=doy).dropna()

# --- 5. WINDOW-day climatology, wrapping at the year boundary --------------
half = WINDOW // 2
clim = {}
for d in range(1, 366):
    w = [(d + k - 1) % 365 + 1 for k in range(-half, half + 1)]
    v = r[r.index.isin(w)].values
    clim[d] = (v.mean(), np.percentile(v, 10), np.percentile(v, 90))
clim = pd.DataFrame(clim, index=['mean', 'p10', 'p90']).T
print(f'values per day-of-year window: ~{len(r) * WINDOW // 365}')

# --- 5b. precipitation: wet-day frequency and intensity --------------------
WET = 1.0   # mm, ETCCDI wet-day threshold
if VAR == 'precip_mm':
    wet = {}
    for d in range(1, 366):
        w  = [(d + k - 1) % 365 + 1 for k in range(-half, half + 1)]
        v  = r[r.index.isin(w)].values
        vw = v[v >= WET]
        if len(vw) >= 20:
            wet[d] = ((v >= WET).mean(), vw.mean(), np.percentile(vw, 90))
        else:
            wet[d] = ((v >= WET).mean(), np.nan, np.nan)
    wet = pd.DataFrame(wet, index=['wet_freq', 'wet_mean', 'wet_p90']).T
    wet = (pd.concat([wet] * 3)
             .rolling(SMOOTH, center=True, min_periods=SMOOTH // 2)
             .mean().iloc[365:730])
    wet.index = range(1, 366)
    print(wet.describe())

# --- 6. circular SMOOTH-day smoothing ----------------------------------------
clim = pd.concat([clim] * 3).rolling(SMOOTH, center=True).mean().iloc[365:730]
clim.index = range(1, 366)

if VAR == 'precip_mm':
    clim = clim[['mean']]                                     # drop p10/p90
    monthly = (s.groupby(s.index.month).mean() * 30.44).round(1)
    print('mean monthly precipitation (mm):'); print(monthly.to_string())

print(clim.describe())
#clim.to_csv(f'climatology_{VAR}_{TAG}.csv', index_label='doy')

In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), constrained_layout=True)

# top: raw daily series with a 1-year running mean
a1.plot(s.index, s.values, lw=0.3, color='0.6', label='daily')
a1.plot(s.rolling(365, center=True, min_periods=300).mean(),
        lw=1.2, color='C0', label='365-day mean')
a1.set_ylabel(UNIT)
a1.set_title(f'{info["label"]} [{TAG}]')
a1.legend(loc='upper left')

# bottom: annual means with the fitted trend
yrs = ann.index.year
a2.plot(yrs, ann.values, 'o-', ms=3, color='C0')
if res is not None:
    a2.plot(yrs, res.intercept + res.slope * yrs, '--', color='C3',
            label=f'trend {res.slope*10:.3f} {UNIT} per decade')
    a2.legend(loc='upper left')
a2.set_ylabel(f'annual mean ({UNIT})')
plt.show()

In [ ]:
yrs = ann.index.year.values
lin = np.polyval(np.polyfit(yrs, ann.values, 1), yrs)                         # straight line
cur = np.polyval(np.polyfit(yrs - yrs.mean(), ann.values, 3), yrs - yrs.mean())  # smooth curve

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)

# top: what each method would remove
a1.plot(yrs, ann.values, 'o-', ms=3, color='0.5', label='annual mean')
a1.plot(yrs, lin, '--', color='C3', label='straight line')
a1.plot(yrs, cur, '-',  color='C2', label='smooth curve')
a1.set_ylabel(UNIT); a1.legend(loc='upper left')
a1.set_title('What each detrend would remove')

# bottom: what is left after removing it
a2.axhline(0, color='k', lw=0.6)
a2.plot(yrs, ann.values - lin, 'o-', ms=3, color='C3', label='after straight-line detrend')
a2.plot(yrs, ann.values - cur, 'o-', ms=3, color='C2', label='after curve detrend')
a2.set_ylabel(f'residual ({UNIT})'); a2.legend(loc='upper left')
plt.show()

In [ ]:
p90 = clim['p90'].reindex(doy365(s.index)).values
exc = pd.Series(s.values > p90, index=s.index)
print(exc.groupby((exc.index.year // 10) * 10).mean().round(3).to_string())

In [ ]:
sst_e, _ = load_series('sst_c')
sst_h, _ = load_series('temp_surface_c')

print('\nERA5 SST:');  print(sst_e.describe().round(2))
print('\nHYCOM SST:'); print(sst_h.describe().round(2))

print('\nHYCOM days with data per year:')
print(sst_h.groupby(sst_h.index.year).count().to_string())

both = pd.DataFrame({'era5': sst_e, 'hycom': sst_h}).dropna()
print('\noverlap:', both.index[0].date(), 'to', both.index[-1].date(), len(both), 'days')
print('mean difference (HYCOM - ERA5):', round((both.hycom - both.era5).mean(), 2), '°C')
print('daily correlation:', round(both.era5.corr(both.hycom), 2))

In [ ]:
d = both.hycom - both.era5
print(d.groupby(d.index.year).agg(['count', 'mean']).round(2).to_string())

In [ ]:
u, _  = load_series('wind_u_ms')
v, _  = load_series('wind_v_ms')
ws, _ = load_series('wind_speed_ms')

vec = np.sqrt(u**2 + v**2)          # speed of the daily-average wind
diff = ws - vec

print('\nwind_speed_ms mean:', round(ws.mean(), 2), 'm/s')
print('speed of average u,v mean:', round(vec.mean(), 2), 'm/s')
print('fraction of days where they are equal:', round((diff.abs() < 0.01).mean(), 3))
print('fraction of days where wind_speed_ms is larger:', round((diff > 0.01).mean(), 3))

In [ ]:
from scipy import stats

hmax, _ = load_series('hs_max_m')
yr = hmax.groupby(hmax.index.year)
storms = pd.DataFrame({
    'annual_max_m': yr.max().round(2),
    'days_over_2m': yr.apply(lambda x: (x > 2).sum()),
})
print(storms.to_string())
print('\ntrend in annual max:   ', stats.kendalltau(storms.index, storms['annual_max_m']))
print('trend in days over 2 m:', stats.kendalltau(storms.index, storms['days_over_2m']))

In [ ]:
sal, _ = load_series('salinity_surface_psu')

print('\n', sal.describe().round(2))
print('\nper year:')
print(sal.groupby(sal.index.year).agg(['count', 'mean']).round(2).to_string())

print('\nmean 1993-2012 (old HYCOM version):', round(sal['1993':'2012'].mean(), 2), 'PSU')
print('mean 2019-2024 (new HYCOM version):', round(sal['2019':'2024'].mean(), 2), 'PSU')

In [ ]:
print(sal[sal < 35])

In [ ]:
u, _       = load_series('wind_u_ms')
v, _       = load_series('wind_v_ms')
wavedir, _ = load_series('wave_dir_deg')
hmax, _    = load_series('hs_max_m')

wdir = (270 - np.degrees(np.arctan2(v, u))) % 360      # direction the wind comes FROM

names = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
def sector(d):
    d = d.dropna()
    return pd.Series(np.array(names)[(((d + 22.5) % 360) // 45).astype(int)], index=d.index)

def table(d):
    sec = sector(d)
    t = pd.crosstab(sec.index.month, sec, normalize='index') * 100
    return t.reindex(columns=names, fill_value=0).round(0)

print('WIND: % of days from each direction, per month')
print(table(wdir).to_string())
print('\nWAVES: % of days from each direction, per month')
print(table(wavedir).to_string())

storm = sector(wavedir)[hmax > 2]
print('\nSTORM WAVES (max height > 2 m): % from each direction')
print((storm.value_counts(normalize=True) * 100).reindex(names, fill_value=0).round(0).to_string())

In [ ]:
HURDAT = '/Users/sophiethomas/Desktop/MDP/Climatology/hurdat2_atlantic.txt'
SITE = (21.165, -90.03)     # Sisal
RADIUS_KM = 100
YEAR0, YEAR1 = 1950, 2025   # same period as the ERA5-Land baseline

# --- read HURDAT2: one row per 6-hourly track point -------------------------
rows, sid, name = [], None, None
with open(HURDAT) as f:
    for line in f:
        p = [x.strip() for x in line.split(',')]
        if p[0].startswith('AL'):
            sid, name = p[0], p[1]
            continue
        lat = float(p[4][:-1]) * (1 if p[4][-1] == 'N' else -1)
        lon = float(p[5][:-1]) * (-1 if p[5][-1] == 'W' else 1)
        rows.append((sid, name, pd.Timestamp(p[0] + p[1]), p[3], lat, lon, int(p[6])))
trk = pd.DataFrame(rows, columns=['id', 'name', 'time', 'status', 'lat', 'lon', 'wind_kt'])
trk = trk[trk['time'].dt.year.between(YEAR0, YEAR1)]

# --- interpolate to hourly: a fast storm can cross the circle between 6-hourly points
def hourly(g):
    g = g.drop_duplicates('time').set_index('time')
    idx = pd.date_range(g.index[0], g.index[-1], freq='h')
    num = g[['lat', 'lon', 'wind_kt']].reindex(g.index.union(idx)).interpolate('time').reindex(idx)
    num['status'] = g['status'].reindex(idx, method='ffill')
    num['name'] = g['name'].iloc[0]
    return num
h = pd.concat({k: hourly(g) for k, g in trk.groupby('id')}, names=['id', 'time']).reset_index()

# --- distance to Sisal ------------------------------------------------------
la1, lo1 = np.radians(SITE)
la2, lo2 = np.radians(h['lat']), np.radians(h['lon'])
h['km'] = 6371 * 2 * np.arcsin(np.sqrt(np.sin((la2 - la1) / 2)**2
                                       + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2)**2))

# --- tropical/subtropical systems of at least tropical-storm strength (34 kt)
near = h[(h['km'] <= RADIUS_KM) & h['status'].isin(['TS', 'HU', 'SS']) & (h['wind_kt'] >= 34)]

def category(kt):
    for lim, c in [(137, 'Cat 5'), (113, 'Cat 4'), (96, 'Cat 3'), (83, 'Cat 2'), (64, 'Cat 1')]:
        if kt >= lim:
            return c
    return 'TS'

storms = (near.sort_values('km').groupby('id')
              .agg(name=('name', 'first'), date=('time', 'first'),
                   closest_km=('km', 'min'), max_wind_kt=('wind_kt', 'max'))
              .sort_values('date'))
storms['date'] = storms['date'].dt.date
storms[['closest_km', 'max_wind_kt']] = storms[['closest_km', 'max_wind_kt']].round(0)
storms['category'] = storms['max_wind_kt'].apply(category)   # strongest WITHIN the circle, not at Sisal
print(f'Storms within {RADIUS_KM} km of Sisal, {YEAR0}-{YEAR1}: {len(storms)}')
print(storms.to_string())

# --- counts per year and per decade -----------------------------------------
yrs = pd.to_datetime(storms['date']).dt.year
is_hu = storms['category'] != 'TS'
per_year = pd.DataFrame({'all': yrs.value_counts(), 'hurricanes': yrs[is_hu].value_counts()}
                        ).reindex(range(YEAR0, YEAR1 + 1)).fillna(0).astype(int)
dec = (per_year.index // 10) * 10
print('\nstorms per year, by decade:')
print((per_year.groupby(dec).sum() / per_year.groupby(dec).size().values[:, None]).round(2).to_string())
print('\nmean per year, all:', round(per_year['all'].mean(), 2),
      '  hurricanes:', round(per_year['hurricanes'].mean(), 2))
print('trend, all storms:', stats.kendalltau(per_year.index, per_year['all']))
print('trend, hurricanes:', stats.kendalltau(per_year.index, per_year['hurricanes']))

In [ ]:
URL_P = 'https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/MXM00076593.csv'
gp = pd.read_csv(URL_P, parse_dates=['DATE'])
print('columns:', list(gp.columns))

q = gp['TMIN_ATTRIBUTES'].fillna(',,,').str.split(',').str[1]
tmin_p = pd.Series((gp['TMIN'].where(q == '') / 10).values, index=gp['DATE'])   # tenths of °C -> °C
tmin_p = tmin_p.reindex(pd.date_range(tmin_p.index.min(), tmin_p.index.max(), freq='D'))

print(tmin_p.groupby(tmin_p.index.year).agg(['count', 'mean']).round(1).to_string())

In [ ]:
emin, _ = load_series('t2m_min_c', 'c1')

d = (tmin_p - emin).dropna()['2000':]                       # Progreso minus ERA5, same days only
d = d - d.groupby(d.index.month).transform('mean')          # remove the normal seasonal difference

print(d.groupby(d.index.year).agg(['count', 'mean']).round(2).to_string())

In [ ]:
import matplotlib.pyplot as plt

p90 = clim['p90'].reindex(doy365(s.index)).values
exc = pd.Series(s.values > p90, index=s.index)
yearly = exc.groupby(exc.index.year).mean() * 100                 # % of nights per year
dec = yearly.groupby((yearly.index // 10) * 10).mean()            # decade means

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(yearly.index, yearly.values, color='0.75', label='per year')
for i, (d0, v) in enumerate(dec.items()):
    d1 = min(d0 + 9, yearly.index.max())
    ax.hlines(v, d0 - 0.4, d1 + 0.4, color='C3', lw=2.5,
              label='decade mean' if i == 0 else None)
ax.axhline(10, color='k', ls='--', lw=1, label='expected without change (10%)')
ax.set_ylabel('% of nights above p90')
ax.set_title('Warm nights at Sisal (ERA5-Land daily minimum above its 1950–2025 p90)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(yearly.round(1).to_string())

In [ ]:
import xarray, netCDF4
print('xarray', xarray.__version__, ' netCDF4', netCDF4.__version__)

In [ ]:
import glob
import xarray as xr

SL_DIR = '/Users/sophiethomas/Desktop/MDP/Climatology/Baseline/sealevel/'
files = sorted(glob.glob(SL_DIR + '**/*.nc', recursive=True))
print(len(files), 'files')
print('first:', files[0])
print('last: ', files[-1])

ds = xr.open_dataset(files[0])
print(ds)

# sea level anomaly in a ~1° box around Sisal (land cells show as NaN)
lon = ds['longitude']
lo = (-90.6, -89.6) if float(lon.min()) < 0 else (269.4, 270.4)
box = ds['sla'].sel(latitude=slice(20.9, 21.9), longitude=slice(*lo)).squeeze()
print(box.round(3).to_pandas())

In [ ]:
CELLS = {'coast': (21.125, -90.125), 'offshore': (21.375, -90.125)}

rows = []
for f in files:
    with xr.open_dataset(f) as d:
        r = {'date': pd.Timestamp(d['time'].values[0])}
        for k, (la, lo_) in CELLS.items():
            r[k] = float(d['sla'].sel(latitude=la, longitude=lo_, method='nearest').squeeze())
        rows.append(r)

sl = pd.DataFrame(rows).set_index('date') * 1000          # m -> mm
sl.to_csv(SL_DIR + '../sealevel_sisal_monthly.csv')

print(sl.describe().round(1))
print('\nmissing months:', sl.isna().sum().to_dict())

t = ((sl.index - sl.index[0]).days / 365.25).values
for k in CELLS:
    a = sl[k] - sl[k].groupby(sl.index.month).transform('mean')   # remove seasonal cycle
    ok = a.notna().values
    print(f'{k}: trend {np.polyfit(t[ok], a.values[ok], 1)[0]:.2f} mm/yr (1993-2025)')
print('correlation coast vs offshore:', round(sl['coast'].corr(sl['offshore']), 3))

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

s_sl = sl['coast']
anom = s_sl - s_sl.groupby(s_sl.index.month).transform('mean')   # seasonal cycle removed
ann = s_sl.resample('YE').mean()
ann.index = ann.index.year

def trend(y0, y1):
    x = ann.loc[y0:y1]
    r = stats.linregress(x.index, x.values)
    return r.slope, 1.96 * r.stderr, r.intercept

print('trends from annual means (95%):')
for y0, y1 in [(1993, 2025), (1993, 2011), (2012, 2025), (2002, 2011), (2012, 2021)]:
    b, e, _ = trend(y0, y1)
    print(f'  {y0}-{y1}: {b:5.1f} ± {e:.1f} mm/yr')
print('Mann-Kendall 1993-2025:', stats.kendalltau(ann.index, ann.values))

print('\nseasonal cycle (mm, mean per calendar month):')
print(s_sl.groupby(s_sl.index.month).mean().round(0).to_string())
print('\nlargest anomalies after removing the seasonal cycle (mm):')
print(anom.nlargest(5).round(0).to_string())

fig, ax = plt.subplots(2, 1, figsize=(11, 7))
ax[0].plot(anom.index, anom, color='0.6', lw=0.8, label='monthly anomaly')
ax[0].plot(anom.index, anom.rolling(12, center=True).mean(), color='C0', label='12-month mean')
ax[0].set_ylabel('mm'); ax[0].legend(loc='upper left')
ax[0].set_title('Sea level anomaly at Sisal (satellite altimetry, seasonal cycle removed)')

ax[1].plot(ann.index, ann.values, 'o-', color='C0', ms=3, label='annual mean')
for (y0, y1), c in [((1993, 2025), 'k'), ((1993, 2011), 'C2'), ((2012, 2025), 'C3')]:
    b, _, i = trend(y0, y1)
    yrs = np.arange(y0, y1 + 1)
    ax[1].plot(yrs, i + b * yrs, '--', color=c, label=f'{y0}-{y1}: {b:.1f} mm/yr')
ax[1].set_ylabel('mm (relative to 1993-2012)'); ax[1].legend(loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt

SL_DIR = '/Users/sophiethomas/Desktop/MDP/Climatology/Baseline/sealevel/'

# one colour per variable, used in every report figure
# one shade per variable, taken from the IPCC AR6 WGI colour maps (waves: IPCC 'misc' map)
COLORS = {'tmax': '#aa3a3b', 'tmin': '#296a9b', 'sst': '#de764f', 'rain': '#288178',
          'wind': '#32801f', 'wave': '#7954bb', 'sea': '#234a8c', 'storm': '#555555'}
MONTHS = ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D']
m = np.arange(1, 13)

def monthly_band(s):
    """mean, p10 and p90 of the daily values in each calendar month"""
    g = s.dropna().groupby(s.dropna().index.month)
    return g.mean(), g.quantile(0.1), g.quantile(0.9)

tmax, _ = load_series('t2m_max_c', 'c1')
tmin, _ = load_series('t2m_min_c', 'c1')
rain, _ = load_series('precip_mm', 'c1')
sst, _  = load_series('sst_c')
wind, _ = load_series('wind_speed_ms')
hs, _   = load_series('hs_mean_m')
sea = pd.read_csv(SL_DIR + '../sealevel_sisal_monthly.csv', parse_dates=['date'], index_col='date')['coast']

rain_month = rain.resample('MS').sum()
rain_month = rain_month.groupby(rain_month.index.month).mean()          # mm per month
sea_cycle = sea.groupby(sea.index.month).mean()
sea_cycle = sea_cycle - sea_cycle.mean()                                # mm relative to annual mean

# months of the 18 storms within 100 km (HURDAT2, 1950-2025; see storm table)
storm_months = [8, 9, 9, 10, 9, 10, 8, 6, 9, 9, 10, 9, 7, 6, 10, 8, 7, 10]
storm_count = pd.Series(storm_months).value_counts().reindex(m, fill_value=0)

fig, ax = plt.subplots(7, 1, figsize=(7.5, 11), sharex=True,
                       gridspec_kw={'height_ratios': [1.4, 1, 1, 1, 1, 1, 0.7]})

def band(a, s, c, label):
    mu, lo, hi = monthly_band(s)
    a.fill_between(m, lo, hi, color=c, alpha=0.18, lw=0)
    a.plot(m, mu, color=c, lw=2, marker='o', ms=4, label=label)

band(ax[0], tmax, COLORS['tmax'], 'daily max')
band(ax[0], tmin, COLORS['tmin'], 'daily min')
ax[0].set_ylabel('Air temp.\n(°C)')
ax[0].legend(loc='lower center', ncol=2, frameon=False, fontsize=8)

ax[1].bar(m, rain_month.values, color=COLORS['rain'], width=0.7)
ax[1].set_ylabel('Rainfall\n(mm/month)')

band(ax[2], sst, COLORS['sst'], 'SST');   ax[2].set_ylabel('SST\n(°C)')
band(ax[3], wind, COLORS['wind'], 'wind'); ax[3].set_ylabel('Wind speed\n(m/s)')
band(ax[4], hs, COLORS['wave'], 'Hs');    ax[4].set_ylabel('Wave height\n(m)')

ax[5].axhline(0, color='0.6', lw=0.8)
ax[5].plot(m, sea_cycle.values, color=COLORS['sea'], lw=2, marker='o', ms=4)
ax[5].set_ylabel('Sea level\n(mm vs. annual mean)')

ax[6].bar(m, storm_count.values, color=COLORS['storm'], width=0.7)
ax[6].set_ylabel('Storms\n(1950–2025)')
ax[6].set_yticks([0, 3, 6])

for i, a in enumerate(ax):
    a.grid(True, axis='y', color='0.94', lw=0.5, zorder=0)   # light horizontal lines only
    a.set_axisbelow(True)                            # grid behind lines and bars
    a.spines[['top', 'right']].set_visible(False)
    a.yaxis.set_label_coords(-0.1, 0.5)
    a.tick_params(labelsize=8)
    a.yaxis.label.set_size(8.5)
    a.text(0.005, 0.97, f'({"abcdefg"[i]})', transform=a.transAxes,   # panel letter
           ha='left', va='top', fontsize=9, fontweight='bold')
ax[-1].set_xticks(m, MONTHS)
ax[-1].set_xlim(0.4, 12.6)
ax[0].set_title('A year at Sisal: mean seasonal cycle (shaded band: 10th–90th percentile of daily values)',
                fontsize=9, loc='left')

plt.tight_layout(h_pad=0.4)
fig.savefig(SL_DIR + '../fig1_seasonal_cycle.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

SL_DIR = '/Users/sophiethomas/Desktop/MDP/Climatology/Baseline/sealevel/'
COLORS = {'tmax': '#aa3a3b', 'tmin': '#296a9b', 'sst': '#de764f', 'rain': '#288178',
          'wind': '#32801f', 'wave': '#7954bb', 'sea': '#234a8c', 'storm': '#555555'}

def annual(s):
    g = s.resample('YE')
    a = g.mean()[g.count() >= 329]            # years at least 90% complete
    a.index = a.index.year
    return a

tmax, _ = load_series('t2m_max_c', 'c1')
tmin, _ = load_series('t2m_min_c', 'c1')
sst, _  = load_series('sst_c')
wind, _ = load_series('wind_speed_ms')
hs, _   = load_series('hs_mean_m')
sea = pd.read_csv(SL_DIR + '../sealevel_sisal_monthly.csv', parse_dates=['date'], index_col='date')['coast']
sea_ann = sea.resample('YE').mean(); sea_ann.index = sea_ann.index.year

# storms within 100 km (HURDAT2), one entry per storm
storm_years = [1951, 1955, 1956, 1966, 1967, 1969, 1973, 1974, 1974, 1988,
               1995, 2002, 2005, 2020, 2020, 2021, 2024, 2024]
storms_per_year = pd.Series(storm_years).value_counts().reindex(range(1950, 2026), fill_value=0)

# (series, colour, y-label, panel text, draw trend line?)
PANELS = [
    (annual(tmax), COLORS['tmax'], 'Daily max\n(°C)',      '+0.09 ± 0.04 °C/decade (p < 0.001)', True),
    (annual(tmin), COLORS['tmin'], 'Daily min\n(°C)',      '+0.13 ± 0.03 °C/decade (p < 0.001)', True),
    (annual(sst),  COLORS['sst'],  'SST\n(°C)',            '+0.09 ± 0.07 °C/decade (p = 0.05, borderline)', True),
    (sea_ann,      COLORS['sea'],  'Sea level\n(mm, vs. 1993–2012)',      '+5.3 ± 0.8 mm/yr (p < 0.001)', True),
    (annual(wind), COLORS['wind'], 'Wind speed\n(m/s)',    '−0.04 m/s/decade (p = 0.02; not robust)', False),
    (annual(hs),   COLORS['wave'], 'Wave height\n(m)',     'no trend (p = 0.76)', False),
]

fig, ax = plt.subplots(len(PANELS) + 1, 1, figsize=(7.5, 11), sharex=True)

for i, (a_s, c, ylab, txt, line) in enumerate(PANELS):
    a = ax[i]
    a.plot(a_s.index, a_s.values, color=c, lw=1.5, marker='o', ms=2.5)
    if line:
        r = stats.linregress(a_s.index, a_s.values)
        a.plot(a_s.index, r.intercept + r.slope * a_s.index, '--', color='0.35', lw=1)
    a.set_ylabel(ylab)
    lo, hi = a.get_ylim(); a.set_ylim(lo, hi + 0.35 * (hi - lo))      # headroom for the label
    a.text(0.995, 0.95, txt, transform=a.transAxes, ha='right', va='top', fontsize=8, color='0.25')

ax[-1].bar(storms_per_year.index, storms_per_year.values, color=COLORS['storm'], width=0.8)
ax[-1].set_ylabel('Storms\n(per year)')
ax[-1].set_yticks([0, 1, 2]); ax[-1].set_ylim(0, 2.9)
ax[-1].text(0.995, 0.95, '18 storms in 76 years: too few for a trend test',
            transform=ax[-1].transAxes, ha='right', va='top', fontsize=8, color='0.25')

for i, a in enumerate(ax):
    a.grid(True, axis='y', color='0.94', lw=0.5, zorder=0)
    a.set_axisbelow(True)
    a.spines[['top', 'right']].set_visible(False)
    a.yaxis.set_label_coords(-0.1, 0.5)
    a.tick_params(labelsize=8)
    a.yaxis.label.set_size(8.5)
    a.text(0.005, 0.95, f'({"abcdefg"[i]})', transform=a.transAxes,
           ha='left', va='top', fontsize=9, fontweight='bold')
ax[-1].set_xlim(1948, 2027)
ax[0].set_title('What changed at Sisal: annual means and trends', fontsize=9, loc='left')

plt.tight_layout(h_pad=0.4)
fig.savefig(SL_DIR + '../fig2_annual_trends.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

SL_DIR = '/Users/sophiethomas/Desktop/MDP/Climatology/Baseline/sealevel/'
SEA = '#234a8c'                                       # IPCC sea level colour (same as Fig. 1-2)

sea = pd.read_csv(SL_DIR + '../sealevel_sisal_monthly.csv', parse_dates=['date'], index_col='date')['coast']
anom = sea - sea.groupby(sea.index.month).transform('mean')        # seasonal cycle removed
g = sea.resample('YE')
ann = g.mean()[g.count() == 12]; ann.index = ann.index.year         # complete years only
xm = anom.index.year + (anom.index.month - 0.5) / 12                 # decimal year, monthly

def trend(y0, y1):
    x = ann.loc[y0:y1]
    r = stats.linregress(x.index, x.values)
    return r.slope, 1.96 * r.stderr, r.intercept

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                       gridspec_kw={'height_ratios': [1, 1.2]})

# (a) monthly anomaly with 12-month running mean
ax[0].plot(xm, anom.values, color=SEA, alpha=0.3, lw=0.7, label='monthly')
ax[0].plot(xm, anom.rolling(12, center=True).mean().values, color=SEA, lw=1.8, label='12-month running mean')
ax[0].set_ylabel('Sea level anomaly\n(mm, seasonal cycle removed)')
ax[0].legend(loc='upper left', frameon=False, fontsize=8, bbox_to_anchor=(0.04, 1))

# (b) annual means with trends for the full record and before/after 2012
ax[1].plot(ann.index + 0.5, ann.values, 'o-', color=SEA, ms=3.5, lw=1.5, label='annual mean')
for (y0, y1), style, c in [((1993, 2025), '-', '0.55'), ((1993, 2011), '--', '0.2'), ((2012, 2025), '--', '0.2')]:
    b, e, i = trend(y0, y1)
    yrs = np.arange(y0, y1 + 1)
    ax[1].plot(yrs + 0.5, i + b * yrs, style, color=c, lw=1.2, label=f'{y0}–{y1}: {b:.1f} ± {e:.1f} mm/yr')
ax[1].set_ylabel('Sea level\n(mm, vs. 1993–2012)')
ax[1].legend(loc='upper left', frameon=False, fontsize=8, bbox_to_anchor=(0.04, 1))

for i, a in enumerate(ax):
    a.grid(True, axis='y', color='0.94', lw=0.5, zorder=0)
    a.set_axisbelow(True)
    a.spines[['top', 'right']].set_visible(False)
    a.yaxis.set_label_coords(-0.09, 0.5)
    a.tick_params(labelsize=8)
    a.yaxis.label.set_size(8.5)
    a.text(0.005, 0.97, f'({"ab"[i]})', transform=a.transAxes,
           ha='left', va='top', fontsize=9, fontweight='bold')
ax[1].set_xlim(1992.5, 2026.5)
ax[0].set_title('Sea level at Sisal (satellite altimetry, 21.125°N 90.125°W)', fontsize=9, loc='left')

plt.tight_layout(h_pad=0.6)
fig.savefig(SL_DIR + '../fig3_sea_level.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import cartopy
print(cartopy.__version__)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patheffects as pe

HURDAT = '/Users/sophiethomas/Desktop/MDP/Climatology/hurdat2_atlantic.txt'   # adjust if needed
SL_DIR = '/Users/sophiethomas/Desktop/MDP/Climatology/Baseline/sealevel/'
SITE = (21.165, -90.03)                     # Sisal (lat, lon)
WAVE = '#7954bb'                            # IPCC 'misc' colour used for waves in Fig. 1-2

# storms within 100 km and their strongest category inside the circle (see storm table)
STORMS = {'AL041951': 'Cat 1', 'AL091955': 'TS', 'AL071956': 'TS', 'AL091966': 'Cat 2',
          'AL131967': 'Cat 1', 'AL261969': 'TS', 'AL081973': 'TS', 'AL011974': 'TS',
          'AL101974': 'Cat 1', 'AL081988': 'Cat 3', 'AL171995': 'TS', 'AL102002': 'Cat 3',
          'AL052005': 'Cat 1', 'AL032020': 'TS', 'AL282020': 'TS', 'AL072021': 'TS',
          'AL022024': 'TS', 'AL142024': 'Cat 5'}
CLASS = {'TS': 'Tropical storm', 'Cat 1': 'Hurricane cat. 1–2', 'Cat 2': 'Hurricane cat. 1–2',
         'Cat 3': 'Major hurricane (cat. 3–5)', 'Cat 5': 'Major hurricane (cat. 3–5)'}
CLASS_COLOR = {'Tropical storm': '#bdbdbd', 'Hurricane cat. 1–2': '#de764f',
               'Major hurricane (cat. 3–5)': '#68352a'}          # IPCC temperature ramp steps

# --- read the tracks of these storms from HURDAT2 ---------------------------
tracks, sid = {}, None
with open(HURDAT) as f:
    for line in f:
        p = [x.strip() for x in line.split(',')]
        if p[0].startswith('AL'):
            sid = p[0]
            continue
        if sid in STORMS:
            lat = float(p[4][:-1]) * (1 if p[4][-1] == 'N' else -1)
            lon = float(p[5][:-1]) * (-1 if p[5][-1] == 'W' else 1)
            tracks.setdefault(sid, []).append((lon, lat))

# --- wave direction sectors: all days vs storm-wave days --------------------
wavedir, _ = load_series('wave_dir_deg')
hmax, _ = load_series('hs_max_m')
d = pd.DataFrame({'dir': wavedir, 'h': hmax}).dropna()
sec = (((d['dir'] + 22.5) % 360) // 45).astype(int)              # 0 = N, 1 = NE, ...
all_pct   = sec.value_counts(normalize=True).reindex(range(8), fill_value=0) * 100
storm_pct = sec[d['h'] > 2].value_counts(normalize=True).reindex(range(8), fill_value=0) * 100

# --- figure -----------------------------------------------------------------
fig = plt.figure(figsize=(11, 5))
proj = ccrs.PlateCarree()
ax0 = fig.add_axes([0.03, 0.08, 0.55, 0.84], projection=proj)
ax0.set_extent([-93.5, -86.5, 18.5, 24.0], crs=proj)
ax0.add_feature(cfeature.LAND.with_scale('10m'), facecolor='0.95', edgecolor='none')
ax0.add_feature(cfeature.COASTLINE.with_scale('10m'), edgecolor='0.4', lw=0.6)

for order in ['Tropical storm', 'Hurricane cat. 1–2', 'Major hurricane (cat. 3–5)']:   # strongest on top
    for k, pts in tracks.items():
        cl = CLASS[STORMS[k]]
        if cl != order:
            continue
        lon, lat = zip(*pts)
        ax0.plot(lon, lat, color=CLASS_COLOR[cl], lw=2 if 'Major' in cl else 1.3,
                 transform=proj, zorder=3 if 'Major' in cl else 2)

# 100 km circle (geodesic) and Sisal
ang = np.linspace(0, 2 * np.pi, 200)
clat = SITE[0] + (100 / 111.2) * np.sin(ang)
clon = SITE[1] + (100 / (111.2 * np.cos(np.radians(SITE[0])))) * np.cos(ang)
ax0.plot(clon, clat, color='k', lw=1, ls='--', transform=proj, zorder=4)
ax0.plot(SITE[1], SITE[0], marker='*', ms=11, color='k', mec='white', transform=proj, zorder=5)
ax0.text(SITE[1] + 0.12, SITE[0] - 0.25, 'Sisal', fontsize=9, transform=proj, zorder=6,
         path_effects=[pe.withStroke(linewidth=2.5, foreground='white')])

LABEL_POS = {'Gilbert 1988': (-88.2, 20.45), 'Isidore 2002': (-89.5, 23.25), 'Milton 2024': (-88.6, 22.8)}
for name, (x, y) in LABEL_POS.items():     # label the three major hurricanes next to their tracks
    ax0.text(x, y, name, fontsize=8, color=CLASS_COLOR['Major hurricane (cat. 3–5)'], transform=proj, zorder=6,
             path_effects=[pe.withStroke(linewidth=2.5, foreground='white')])

for cl, c in CLASS_COLOR.items():
    ax0.plot([], [], color=c, lw=2, label=cl)
ax0.plot([], [], color='k', lw=1, ls='--', label='100 km from Sisal')
ax0.legend(loc='lower left', fontsize=8, frameon=True, framealpha=0.9)
gl = ax0.gridlines(draw_labels=True, lw=0.3, color='0.8')
gl.top_labels = gl.right_labels = False
gl.xlabel_style = gl.ylabel_style = {'size': 8}
ax0.set_title('(a) Tropical cyclones passing within 100 km, 1950–2025 (HURDAT2)', fontsize=9, loc='left')

# wave rose (directions waves come FROM)
ax1 = fig.add_axes([0.64, 0.12, 0.33, 0.72], projection='polar')
ax1.set_theta_zero_location('N'); ax1.set_theta_direction(-1)
th = np.radians(np.arange(8) * 45)
w = np.radians(45) * 0.8
ax1.bar(th, all_pct.values, width=w, color=WAVE, alpha=0.3, label='all days', edgecolor='white')
ax1.bar(th, storm_pct.values, width=w * 0.5, color=WAVE, label='storm-wave days (Hs max > 2 m)', edgecolor='white')
ax1.set_xticks(th, ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
ax1.set_rlabel_position(225)                   # radial labels in the empty SW sector
ax1.tick_params(axis='y', labelsize=7)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax1.grid(color='0.88', lw=0.5)
ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), fontsize=8, frameon=False, ncol=1)
ax1.set_title('(b) Wave direction (from), ERA5 1979–2025', fontsize=9, loc='left', pad=18)

fig.savefig(SL_DIR + '../fig4_storms_waves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

OUT = '/Users/sophiethomas/Desktop/MDP/Climatology/Baseline/'
COLORS = {'tmax': '#aa3a3b', 'tmin': '#296a9b', 'rain': '#288178'}
MONTHS = ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D']
m = np.arange(1, 13)

# --- station data (GHCN-Daily Sisal), QC-flagged values removed --------------
URL = 'https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/MXN00031029.csv'
g = pd.read_csv(URL, parse_dates=['DATE'])
idx = pd.date_range(g['DATE'].min(), g['DATE'].max(), freq='D')
st = {}
for el in ['PRCP', 'TMAX', 'TMIN']:
    q = g[f'{el}_ATTRIBUTES'].fillna(',,,').str.split(',').str[1]
    st[el] = pd.Series((g[el].where(q == '') / 10).values, index=g['DATE']).reindex(idx)

# --- ERA5-Land cell c1 --------------------------------------------------------
e_p, _ = load_series('precip_mm', 'c1')
e_x, _ = load_series('t2m_max_c', 'c1')
e_n, _ = load_series('t2m_min_c', 'c1')

# rainfall: 1958-2007, years 1990-91 excluded, same days only
p = pd.DataFrame({'st': st['PRCP'], 'era': e_p}).dropna()
p = p[(p.index.year >= 1958) & ~p.index.year.isin([1990, 1991])]
rain = p.groupby(p.index.month).mean() * 30.44                 # mm/month

# temperature: 1964-1980, same days only
t = pd.DataFrame({'st_x': st['TMAX'], 'era_x': e_x, 'st_n': st['TMIN'], 'era_n': e_n})['1964':'1980'].dropna()
temp = t.groupby(t.index.month).mean()

print('annual rainfall (mm): station', round(rain['st'].sum()), ' ERA5', round(rain['era'].sum()))
print('mean bias max:', round((temp.era_x - temp.st_x).mean(), 1), ' min:', round((temp.era_n - temp.st_n).mean(), 1))

# --- figure -------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))

w = 0.38
ax[0].bar(m - w / 2, rain['st'], width=w, color='white', edgecolor='0.25', lw=0.8, label='Sisal station')
ax[0].bar(m + w / 2, rain['era'], width=w, color=COLORS['rain'], label='ERA5-Land')
ax[0].set_ylabel('Rainfall (mm/month)')
ax[0].set_title('(a) Rainfall, 1958–2007 (1990–91 excluded)', fontsize=9, loc='left')
ax[0].legend(frameon=False, fontsize=8, loc='upper left')

for st_col, era_col, c, name in [('st_x', 'era_x', COLORS['tmax'], 'max'), ('st_n', 'era_n', COLORS['tmin'], 'min')]:
    ax[1].plot(m, temp[st_col], 'o--', color=c, mfc='white', ms=5, lw=1.2, label=f'daily {name}, station')
    ax[1].plot(m, temp[era_col], 'o-', color=c, ms=4, lw=2, label=f'daily {name}, ERA5-Land')
ax[1].set_ylabel('Air temperature (°C)')
ax[1].set_title('(b) Daily max and min temperature, 1964–1980', fontsize=9, loc='left')
ax[1].legend(frameon=False, fontsize=7.5, loc='lower center', ncol=2)
lo, hi = ax[1].get_ylim(); ax[1].set_ylim(lo - 0.25 * (hi - lo), hi)       # room for the legend

for a in ax:
    a.set_xticks(m, MONTHS)
    a.set_xlim(0.4, 12.6)
    a.grid(True, axis='y', color='0.94', lw=0.5, zorder=0)
    a.set_axisbelow(True)
    a.spines[['top', 'right']].set_visible(False)
    a.tick_params(labelsize=8)
    a.yaxis.label.set_size(8.5)

plt.tight_layout(w_pad=2)
fig.savefig(OUT + 'figA1_validation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
SL_DIR = '/Users/sophiethomas/Desktop/MDP/Climatology/Baseline/sealevel/'

# Progreso tide gauge, PSMSL station 690, monthly mean sea level (mm); -99999 = missing
tg = pd.read_csv('https://psmsl.org/data/obtaining/rlr.monthly.data/690.rlrdata',
                 sep=';', header=None, names=['t', 'mm', 'missing_days', 'flag'])
tg = tg[tg['mm'] != -99999].copy()
tg['year'] = tg['t'].astype(int)
tg['month'] = np.floor((tg['t'] - tg['year']) * 12).astype(int) + 1

# seasonal cycle: deviation from each year's mean, only years with at least 10 months
full = tg.groupby('year')['mm'].transform('count') >= 10
tg = tg[full]
tg['dev'] = tg['mm'] - tg.groupby('year')['mm'].transform('mean')
print('tide gauge years used:', tg['year'].nunique(), f"({tg['year'].min()}-{tg['year'].max()})")

# same for the satellite series
sea = pd.read_csv(SL_DIR + '../sealevel_sisal_monthly.csv', parse_dates=['date'], index_col='date')['coast']
sat = sea - sea.groupby(sea.index.year).transform('mean')

cmp = pd.DataFrame({'tide_gauge': tg.groupby('month')['dev'].mean(),
                    'satellite':  sat.groupby(sat.index.month).mean()}).round(0)
print(cmp.to_string())
print('\nrange (max - min), mm:', (cmp.max() - cmp.min()).to_dict())
print('correlation of the two cycles:', round(cmp['tide_gauge'].corr(cmp['satellite']), 2))